# REapk DEX Engine: Playground

A hands-on tour of the REapk DEX engine on a real app. It explains the DEX file format and then exercises the engine end to end: reading the constant pools, disassembling to smali, reassembling and proving a byte-identical round trip, force-return patching a method, bypassing SSL pinning across a whole app and re-signing it, interning a brand-new string with a whole-DEX rewrite, and printing a full smali listing.

Point it at your own APK by setting the `REAPK_TEST_APK` environment variable before launching Jupyter. Everything here is for apps you are authorized to test. Every cell is guarded, so with no APK set the notebook runs clean and just prints a reminder.

## Setup

This notebook needs the `reapk` package importable in this kernel. If the import cell fails with `ModuleNotFoundError`, install it into the kernel's own interpreter with the `%pip` magic (not `!pip`), then restart the kernel (Kernel → Restart):

```python
%pip install -e "C:/Users/dildev/Documents/github/apkx"
```

Use the path to your checkout, or `%pip install -e .` if you launched Jupyter from the repo root.

In [ ]:
import reapk
print("reapk", reapk.__version__)

In [ ]:
import os
APK = os.environ.get("REAPK_TEST_APK")
if not APK:
    print("REAPK_TEST_APK is not set. Set it to an .apk path and restart the")
    print("kernel to run the cells below. They are all guarded, so this notebook")
    print("still executes top to bottom without one.")
else:
    print("using", APK)

## 1 · What a DEX is

A `.dex` (Dalvik Executable) holds all the compiled code in an APK. Its layout is fixed: a 0x70-byte header, then five id tables (strings, types, protos, fields, methods), then the class definitions, then a data section holding the actual blobs (string bytes, code, class data), and finally a map_list that indexes every section. The id tables are pure indices into the string pool, so almost everything in a DEX bottoms out as a string. We open an APK and read these counts straight from the header of its first dex.

In [ ]:
if APK:
    from reapk import Apk
    from reapk.playground import hexdump

    apk = Apk.open(APK)
    print("package:", apk.manifest.package, "| dex files:", len(apk.dex))
    dex = apk.dex[0]
    print("strings:", dex.n_str, " types:", dex.n_type, " protos:", dex.n_proto)
    print("fields: ", dex.n_field, " methods:", dex.n_method, " classes:", dex.n_class)
    print()
    print("header (first 0x70 bytes):")
    print(hexdump(dex.d, 0, 0x70))

## 2 · The constant pools

Every type, field, and method is stored once and referenced by index. `dex.string(i)` reads the string pool (decoding Dalvik's modified UTF-8), `dex.type(i)` resolves a type descriptor, and `dex.method_full(i)` returns a method as (class, name, return type, parameters). Deduplicating into pools is why the writer has to remap every index when anything new is interned (section 7).

In [ ]:
if APK:
    print("first strings:", [dex.string(i) for i in range(min(6, dex.n_str))])
    print("first types:  ", [dex.type(i) for i in range(min(6, dex.n_type))])
    cls, name, ret, params = dex.method_full(0)
    print("method[0]:", cls, name, "params=", params, "ret=", ret)

## 3 · Disassembling a method to smali

Each Dalvik instruction is one or more 16-bit words, with the opcode in the low byte of the first word. The disassembler decodes those words into smali. We walk the first dex, take the first method that has a code body, and disassemble it.

In [ ]:
if APK:
    from reapk.dex import disassemble
    from reapk.playground import show_smali

    hit = None
    for c in dex.classes():
        for m in dex.class_methods(c["cdata"]):
            if m["code_off"]:
                hit = (c["name"], m)
                break
        if hit:
            break

    if hit:
        cls, m = hit
        ci, lines = disassemble(dex, m["code_off"])
        print(cls, "->", m["name"] + m["proto"], "| registers:", ci["regs"])
        print(show_smali(lines[:30]))

## 4 · Round trip: smali back to bytecode

The assembler is the disassembler's inverse. It resolves smali operand references back to pool indices using the dex, then encodes each instruction. If disassembly and assembly agree, re-encoding a method yields byte-identical words. We scan real methods until one round-trips exactly (skipping anything the assembler does not yet handle).

In [ ]:
if APK:
    import struct
    from reapk.dex import assemble, Assembler, AsmSkip

    asm = Assembler(dex)
    checked = matched = 0
    example = None
    for c in dex.classes():
        for m in dex.class_methods(c["cdata"]):
            if not m["code_off"]:
                continue
            ci, lines = disassemble(dex, m["code_off"])
            original = [struct.unpack_from("<H", dex.d, ci["insns_off"] + 2 * i)[0]
                        for i in range(ci["insns_size"])]
            try:
                words = assemble(asm, lines)
            except AsmSkip:
                continue
            checked += 1
            if words == original:
                matched += 1
                if example is None:
                    example = (c["name"], m["name"] + m["proto"], len(original))
            if checked >= 300:
                break
        if checked >= 300:
            break
    print("round-tripped %d/%d methods byte-for-byte" % (matched, checked))
    if example:
        print("example: %s -> %s (%d words)" % example)

## 5 · Patching a method: force-return

`build_dex(dex, replacements={code_off: new_code_item})` re-emits the whole DEX with one or more method bodies swapped, recomputing offsets and the map_list. To force a value return, build a tiny body and drop it in. Here we find a boolean-returning method and force it to return false, then re-parse to prove the change took and the file is still valid.

In [ ]:
if APK:
    from reapk.dex import build_return_insns, build_code_item, build_dex, DexFile

    target = None
    for c in dex.classes():
        for m in dex.class_methods(c["cdata"]):
            if m["code_off"] and m["proto"].endswith(")Z"):
                ci = dex.code_insns(m["code_off"])
                if ci["regs"] >= 1:
                    target = (c["name"], m, ci)
                    break
        if target:
            break

    if target:
        cls, m, ci = target
        before = disassemble(dex, m["code_off"])[1]
        words = build_return_insns("Z", "false", ci["regs"])
        new_code = build_code_item(ci["regs"], ci["ins"], 0, words)
        dex2 = DexFile(build_dex(dex, replacements={m["code_off"]: new_code}))
        m2 = dex2.find_method(cls, m["name"])
        after = disassemble(dex2, m2["code_off"])[1]
        print("patched:", cls, "->", m["name"] + m["proto"])
        print("before:", before[:1], "...")
        print("after: ", after)

## 6 · SSL-pinning bypass and re-sign

For apps you are authorized to test, an intercepting proxy is blocked by certificate pinning and hostname verification done in code. The same force-return primitive neutralizes it: scan every dex for the usual TLS sinks (okhttp's `CertificatePinner.check`, a trust manager's `checkServerTrusted` / `checkClientTrusted`, and hostname verifiers), and make each one a no-op (return void, or return true for hostname checks). The `verify` rule is kept narrow (hostname verifiers and methods that take an `SSLSession`) so unrelated signature checks are left alone.

In [ ]:
if APK:
    import re
    import zipfile
    from reapk.dex import build_return_insns, build_code_item, build_dex

    def find_pin_sinks(dex):
        repl, hits = {}, []
        for c in dex.classes():
            cname = c["name"]
            for m in dex.class_methods(c["cdata"]):
                if not m["code_off"]:
                    continue
                nm, proto = m["name"], m["proto"]
                ci = dex.code_insns(m["code_off"])
                words = None
                if nm == "check" and "CertificatePinner" in cname and proto.endswith(")V"):
                    words = build_return_insns("V", "void", ci["regs"])       # pin check -> no-op
                elif nm in ("checkServerTrusted", "checkClientTrusted") and proto.endswith(")V"):
                    words = build_return_insns("V", "void", ci["regs"])       # trust all certs
                elif (nm == "verify" and proto.endswith(")Z") and ci["regs"] >= 1
                      and ("HostnameVerifier" in cname or "Ljavax/net/ssl/SSLSession;" in proto)):
                    words = build_return_insns("Z", "true", ci["regs"])       # hostname verify -> true
                if words is not None:
                    repl[m["code_off"]] = build_code_item(ci["regs"], ci["ins"], 0, words)
                    hits.append(cname + "->" + nm + proto)
        return repl, hits

    names = sorted(n for n in zipfile.ZipFile(APK).namelist()
                   if re.fullmatch(r"classes\d*\.dex", n))
    patched, hits = {}, []
    for name, d in zip(names, apk.dex):
        repl, h = find_pin_sinks(d)
        if repl:
            patched[name] = build_dex(d, replacements=repl)
            hits += h
    print("neutralized %d SSL/TLS sink(s):" % len(hits))
    for h in hits:
        print("  ", h)

In [ ]:
if APK and patched:
    from reapk import Signer
    from reapk.zipalign import read_zip_entries, stored_entry, write_aligned_zip

    entries = read_zip_entries(open(APK, "rb").read())
    rebuilt = [stored_entry(e["name"], patched[e["name"].decode()])
               if e["name"].decode() in patched else e
               for e in entries]
    out = Signer().sign(write_aligned_zip(rebuilt))
    with open("pinning-bypassed.apk", "wb") as f:
        f.write(out)
    print("wrote pinning-bypassed.apk:", len(out), "bytes (zipaligned + v2/v3 signed)")
    print("re-opens as:", Apk.open("pinning-bypassed.apk").manifest.package)
elif APK:
    print("no pinning sinks matched in this app; nothing to repackage")

## 7 · Interning a new string and whole-DEX rewrite

Adding anything the DEX does not already contain shifts pool indices, so every reference in the file has to be remapped. `Interner` computes those old-to-new maps and `build_dex` applies them across the id tables, code, debug info, and annotations. Here we add one string, rewrite, and confirm both that the new string is present and that an existing method still resolves.

In [ ]:
if APK:
    from reapk.dex import Interner, build_dex, DexFile

    it = Interner(dex)
    it.add_string("patched-by-reapk")
    it.finalize()
    dex2 = DexFile(build_dex(dex, interner=it))
    after = [dex2.string(i) for i in range(dex2.n_str)]
    print("new string present:", "patched-by-reapk" in after)
    print("string pool grew:", dex.n_str, "->", dex2.n_str)
    cls, name, _ret, _p = dex.method_full(0)
    print("existing method still resolves:", dex2.find_method(cls, name) is not None)

## 8 · Full smali listing with `dump_dex`

`dump_dex` walks every class, method, and bytecode body into one organized smali listing. On a real app that's huge, so cap it with `max_classes` / `max_methods_per_class`. It is a disassembly (smali), not Java decompilation.

In [ ]:
if APK:
    from reapk.playground import dump_dex
    print(dump_dex(apk.dex[0], max_classes=2, max_methods_per_class=4))